In [ ]:
import sys, os, requests
sys.path.append('./src/')

import scanpy as sc
import anndata as ad

from metrics import kmeans_ari
from NBVAE_variants import ZINBVAE, ZINBCVAE, ZINBCSVAENA, ZINBCSVAE, ZINBHCSVAENA, ZINBHCSVAE, ZINBDLVAE, ZINBDIVA, ZINBCCVAE
from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import seaborn as sns

from tqdm import trange, tqdm
from umap import UMAP


import torch, pyro
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import pyro.optim as opt

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)


cmap_trt = LinearSegmentedColormap.from_list("cmap", ["#42378C", "#D9A404"])
cmap_ct = ListedColormap(sns.color_palette('colorblind').as_hex())

## Data

In [ ]:
import os, requests

# Static data path, update when necessary
DATA_PATH = "https://www.dropbox.com/scl/fi/zxta2nf00p8a9do907rrv/kang_et_al_perturtbations_preprocessed.h5ad?rlkey=bk4pbuily0349borou6rnvjds&st=yvp1zxjd&dl=1"
NAME = "kang_et_al_perturtbations_preprocessed.h5ad"


# Reorganize param paths
save_path = "./data/" + NAME

# Send download request
headers = {
    "user-agent": "Wget/1.16 (linux-gnu)"
}  # Dropbox checks the agent for some reason
response = requests.get(
    DATA_PATH, headers=headers, stream=True, allow_redirects=True
)

# Get and process response

## Good
if response.status_code == 200:
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Write with progress bar
    with tqdm(
        total=int(response.headers.get("content-length", 0)),
        unit="B",
        unit_scale=True,
    ) as progress_bar:
        with open(save_path, "wb") as f:
            for data in response.iter_content(1024):
                progress_bar.update(len(data))
                f.write(data)

    print(f"Object saved at {save_path}")

## Unexpected
else:
    print(response.__dict__)
    raise Exception("Object not found at URL")

In [ ]:
# This is skipped, but left to describe preprocessing.
# If you need the raw data for anything, the SeuratData package has it. Check here for more details: https://satijalab.org/seurat/archive/v3.2/immune_alignment.html

# if not os.path.exists('./data/kang_et_al_perturtbations_preprocessed.h5ad'):
#     anndata = ad.read_h5ad('./data/kang_et_al_perturtbations.h5ad')
#     anndata.layers['raw'] = anndata.X.copy()
    
#     map_dict = {
#         'B' : 'B', 
#         'B Activated' : 'B', 
#         'CD4 Memory T' : 'CD4 T', 
#         'CD4 Naive T': 'CD4 T', 
#         'CD8 T': 'CD8 T', 
#         'CD14 Mono': 'CD14 Mono',
#         'CD16 Mono': 'CD16 Mono', 
#         'DC': 'DC', 
#         'Eryth': 'Eryth', 
#         'Mk': 'Mk', 
#         'NK': 'NK', 
#         'T activated': 'T', 
#         'pDC': 'DC',
#     }
    
#     anndata.obs['seurat_annotations'] = anndata.obs['seurat_annotations'].astype(object).apply(lambda x: map_dict[x]).astype('category')    
    
#     sc.pp.filter_cells(anndata, min_genes=200)
#     sc.pp.filter_genes(anndata, min_cells=3)
#     sc.pp.normalize_total(anndata)
#     sc.pp.log1p(anndata)
    
#     sc.pp.highly_variable_genes(anndata, n_top_genes=2000)
#     anndata = anndata[:, anndata.var['highly_variable']]
    
#     anndata.write_h5ad('./data/kang_et_al_perturtbations_preprocessed.h5ad')
    
# else:

anndata = ad.read_h5ad('./data/kang_et_al_perturtbations_preprocessed.h5ad')

In [ ]:
import torch.utils.data as utils

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

batch_size=128
x = torch.FloatTensor(anndata.layers['raw'].copy())
y = torch.FloatTensor(anndata.obs['stim'].cat.codes.to_numpy().reshape(-1,1).copy())
y_info = torch.FloatTensor(anndata.obs['seurat_annotations'].cat.codes.to_numpy().reshape(-1,1).copy())

dataset = utils.TensorDataset(x, y, y_info)
train_set, test_set = dataset, dataset
train_set, test_set = utils.TensorDataset(*train_set[:]), utils.TensorDataset(*test_set[:])
train_loader, test_loader  = torch.utils.data.DataLoader(train_set, shuffle=True, batch_size=batch_size),  torch.utils.data.DataLoader(test_set, shuffle=True, batch_size=batch_size)

In [ ]:
reducer = UMAP(random_state=0, metric='correlation')
umap_data = reducer.fit_transform(np.log1p(np.array(test_set[:][0])))

In [ ]:
from pyfonts import load_font

# load font
font = load_font(
   font_url="https://github.com/stevenpetryk/computer-modern/blob/main/src/cmunrm.ttf?raw=true"
)

In [ ]:
plt.figure(figsize=(10,5))

scatter = plt.scatter(umap_data[:, 0], umap_data[:, 1], c=test_set[:][2], cmap=cmap_ct, s=5, alpha=1)


plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)


plt.ylabel('UMAP 2', fontsize=24, font=font)
plt.xlabel('UMAP 1', fontsize=24, font=font)

#leg = plt.legend(handles=scatter.legend_elements()[0], labels=list(anndata.obs['seurat_annotations'].cat.categories), fontsize=18)


plt.show()

In [ ]:
plt.figure(figsize=(10,5))


scatter = plt.scatter(umap_data[:, 0], umap_data[:, 1], c=test_set[:][1], cmap=cmap_trt, s=1)

plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)

plt.ylabel('UMAP 2', font=font, fontsize=24)
plt.xlabel('UMAP 1', font=font, fontsize=24)

#plt.legend(handles=scatter.legend_elements()[0], labels=list(anndata.obs['stim'].cat.categories), fontsize=18)

plt.show()

## CSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvaena = ZINBCSVAENA(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1)
csvaena_trainer = ThresholdPyroTrainer(0, 50, csvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvaena_trainer.train()

In [ ]:
preds = csvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = csvaena_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect('equal')
plt.ylim(-2,7)
plt.xlim(-2,7)
plt.show()

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect('equal')
plt.ylim(-2,7)
plt.xlim(-2,7)
plt.show()

## CSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae = ZINBCSVAE(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1, adversarial_weight=1e2)
csvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1, csvae, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
csvae_trainer.train()
csvae_trainer.save('params/Kang/csvae_kang')

In [ ]:
preds = csvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = csvae_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())


plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

## HCSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvaena = ZINBHCSVAENA(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1)
hcsvaena_trainer = ThresholdPyroTrainer(0, 50, hcsvaena, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
hcsvaena_trainer.train()

In [ ]:
preds = hcsvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = hcsvaena_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

## HCSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvae = ZINBHCSVAE(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, z_kl_weight=1e-4, w_kl_weight=1 ,adversarial_weight=1e2)
hcsvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1 hcsvae, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
hcsvae_trainer.train()

In [ ]:
preds = hcsvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
rho_s = preds['rho'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = hcsvae_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.ylim(-2,7)
plt.xlim(-2,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

## DIVA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

diva = ZINBDIVA(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, kl_weight=1e-4)
diva_trainer = ThresholdPyroTrainer(0, 50, diva, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
diva_trainer.train()

In [ ]:
preds = diva_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = diva_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())

plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.ylim(-7,7)
plt.xlim(-7,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.ylim(-7,7)
plt.xlim(-7,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

## CCVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

ccvae = ZINBCCVAE(2000, [1], latent_dim=10, w_dim=2, num_layers=1, hidden_dim=128, recon_weight=1, kl_weight=1e-4)
ccvae_trainer = ThresholdPyroTrainer(0, 50, ccvae, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
ccvae_trainer.train()

In [ ]:
preds = ccvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0].cpu()

In [ ]:
trace = ccvae_trainer.get_trace('test')
print(-1 * trace.nodes['rec']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons.log1p().mean(dim=0))
plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
z_s = reducer.fit_transform(z_s)

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.axis('off')
plt.ylim(-7,7)
plt.xlim(-7,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.axis('off')
plt.ylim(-7,7)
plt.xlim(-7,7)

plt.gca().set_aspect(1/plt.gca().get_data_ratio())
plt.show()

## DISCoVeR

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = ZINBDLVAE(2000, [1], latent_dim=10, w_dim=10, num_layers=0, hidden_dim=128, recon_weight=9e-1, recon_weight_z=1e-1, w_kl_weight=1e-4, z_kl_weight=1e-4, adversarial_weight=1e2, learnable_prior=False)
dlvae_trainer = AdversarialThresholdPyroTrainer(0, 50, 1, 1, dlvae, train_loader, test_loader, opt.AdamW({"lr": 1e-3}))
dlvae_trainer.train()

In [ ]:
preds = dlvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons_w = preds['rec_w'][0].cpu()
recons_z = preds['rec_z'][0].cpu()

In [ ]:
trace = dlvae_trainer.get_trace('test')
print(-1 * trace.nodes['rec_w']['fn'].log_prob(test_set[:][0].cuda()).mean().item())
print(-1 * trace.nodes['rec_z']['fn'].log_prob(test_set[:][0].cuda()).mean().item())

plt.figure()
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons_w.log1p().mean(dim=0))
plt.scatter(test_set[:][0].log1p().mean(dim=0), recons_z.log1p().mean(dim=0))

plt.gca().axis('square')

plt.figure()
plt.scatter(test_set[:][0].log1p().var(dim=0), recons_w.log1p().var(dim=0))
plt.scatter(test_set[:][0].log1p().var(dim=0), recons_z.log1p().var(dim=0))
plt.gca().axis('square')


plt.show()

In [ ]:
reducer = UMAP()
umap_recon_zs = reducer.fit_transform(torch.log1p(recons_z))
z_s = reducer.fit_transform(z_s)
w_s = reducer.fit_transform(w_s)

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)

plt.ylabel('UMAP 2', font=font, fontsize=24)
plt.xlabel('UMAP 1', font=font, fontsize=24)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(z_s[:, 0], z_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)

plt.ylabel('UMAP 2', font=font, fontsize=24)
plt.xlabel('UMAP 1', font=font, fontsize=24)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][2].numpy(), cmap=cmap_ct, s=1)
plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)

plt.ylabel('UMAP 2', font=font, fontsize=24)
plt.xlabel('UMAP 1', font=font, fontsize=24)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(w_s[:, 0], w_s[:, 1], c=test_set[:][1].numpy(), cmap=cmap_trt, s=1)
plt.gca().set_aspect('equal')
plt.yticks([])
plt.xticks([])
plt.gca().spines[['right', 'top']].set_visible(False)

plt.ylabel('UMAP 2', font=font, fontsize=24)
plt.xlabel('UMAP 1', font=font, fontsize=24)

plt.show()